# 04 — Weighted Matrix Factorization from Scratch (ALS)

Implements the **Hu, Koren & Volinsky (2008)** WMF model with **Alternating Least Squares**,
using only NumPy, and checks it reaches the same ranking quality as the `implicit` library on
identical data.

Pure-NumPy ALS with a per-entity Python loop is fine for tens of thousands of users but not for
all 162K, so this notebook runs on a **random 15,000-user subsample** (the library handles full
scale in nb 05). Both models here see the *exact same* subsample, split and candidates, so the
comparison is apples-to-apples.

In [1]:
import os, sys, time
os.environ['OPENBLAS_NUM_THREADS'] = '1'
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, pandas as pd, scipy.sparse as sp
import recsys_utils as ru
DATA = '../data/ml-25m'
rng = np.random.default_rng(0)

## Build a 15K-user subsample (same prep as nb 02)

In [2]:
ratings = pd.read_csv(f'{DATA}/ratings.csv')
keep = rng.choice(ratings.userId.unique(), size=15000, replace=False)
ratings = ratings[ratings.userId.isin(keep)]
ratings, _ = ru.filter_consensus_bad(ratings)
uc = ratings.userId.value_counts()
ratings = ratings[ratings.userId.isin(uc[uc >= ru.MIN_USER_RATINGS].index)]
ratings, user_ids, movie_ids = ru.reindex(ratings)
n_users, n_items = int(ratings.u.max()+1), int(ratings.i.max()+1)
train, test = ru.leave_one_out_split(ratings)
R = ru.build_confidence_matrix(train, n_users, n_items)
users, cands = ru.build_eval_candidates(test.u.values, test.i.values, R, n_items, seed=0)
print(f'{n_users:,} users | {n_items:,} items | {R.nnz:,} train interactions | '
      f'{len(users):,} eval users')

15,000 users | 25,266 items | 2,239,405 train interactions | 14,952 eval users


## The math (what the code below implements)

Minimise, over user vectors $x_u$ and item vectors $y_i$:

$$\sum_{u,i} c_{ui}\,(p_{ui} - x_u^\top y_i)^2 \;+\; \lambda\Big(\sum_u\|x_u\|^2 + \sum_i\|y_i\|^2\Big)$$

- $p_{ui}=1$ if the user rated movie $i$, else $0$ (**preference**).
- $c_{ui}=1+\alpha\,r_{ui}$ (**confidence** — higher ratings = more confidence in that positive).

The cost is quadratic, so **fixing one factor matrix makes the other solvable in closed form**.
With all $y_i$ fixed, each user vector is:

$$x_u = (Y^\top C^u Y + \lambda I)^{-1}\,Y^\top C^u p(u)$$

The HKV speed-up avoids the dense $n\_items \times n\_items$ matrix $C^u$ by writing
$Y^\top C^u Y = Y^\top Y + Y^\top (C^u - I) Y$, where:
- $Y^\top Y$ is computed **once per sweep** and shared by every user, and
- $(C^u - I)$ is non-zero only on the handful of items the user actually rated.

So each user's update costs work proportional to *their* number of ratings, not the whole catalog.
Then alternate: solve all users, then all items, repeat. No learning rate.

In [3]:
def _solve_factors(R, Y, reg, alpha):
    """One ALS half-sweep: given fixed factors Y, return new factors for every row of R.
    R is a CSR matrix whose stored values are the raw ratings r_ui."""
    f = Y.shape[1]
    YtY = Y.T @ Y                       # shared across all rows (the HKV trick)
    lam = reg * np.eye(f)
    X = np.zeros((R.shape[0], f))
    indptr, indices, data = R.indptr, R.indices, R.data
    for e in range(R.shape[0]):
        s, t = indptr[e], indptr[e+1]
        if s == t:
            continue                    # no ratings -> stays at zero (prior only)
        idx = indices[s:t]
        c = 1.0 + alpha * data[s:t]      # confidence c_ui = 1 + alpha*r
        Ye = Y[idx]                      # (k, f) factors of this row's rated items
        # A = YtY + Ye^T (c-1) Ye + lambda*I ;  b = Ye^T c   (p=1 on observed)
        A = YtY + (Ye * (c - 1.0)[:, None]).T @ Ye + lam
        b = Ye.T @ c
        X[e] = np.linalg.solve(A, b)
    return X

class WMF:
    """Weighted Matrix Factorization (implicit feedback) trained with ALS."""
    def __init__(self, factors=64, regularization=0.05, alpha=40.0, iterations=15, seed=42):
        self.f, self.reg, self.alpha, self.iters, self.seed = (
            factors, regularization, alpha, iterations, seed)
    def fit(self, R):
        rng = np.random.default_rng(self.seed)
        nu, ni = R.shape
        self.user_factors = 0.01 * rng.standard_normal((nu, self.f))
        self.item_factors = 0.01 * rng.standard_normal((ni, self.f))
        Rt = R.T.tocsr()
        for it in range(self.iters):
            self.user_factors = _solve_factors(R,  self.item_factors, self.reg, self.alpha)
            self.item_factors = _solve_factors(Rt, self.user_factors, self.reg, self.alpha)
        return self

## Train the from-scratch model and evaluate

In [4]:
t = time.time()
wmf = WMF(factors=64, regularization=0.05, alpha=ru.ALPHA, iterations=15).fit(R)
print(f'trained from-scratch WMF in {time.time()-t:.1f}s')
scores = ru.score_als(wmf.user_factors.astype(np.float32),
                      wmf.item_factors.astype(np.float32), users, cands)
r_scratch, n_scratch = ru.score_metrics(scores, k=10)
print(f'from-scratch WMF  ->  Recall@10 = {r_scratch:.4f}   NDCG@10 = {n_scratch:.4f}')

trained from-scratch WMF in 48.7s


from-scratch WMF  ->  Recall@10 = 0.9259   NDCG@10 = 0.6970


## Sanity check vs the `implicit` library on the SAME subsample

The library uses confidence `alpha*r` while this implementation uses the paper's `1 + alpha*r`;
the small constant barely matters at `alpha=40`, so the ranking metrics should land very close.
Matching a battle-tested library is the evidence the from-scratch math is correct.

In [5]:
from implicit.cpu.als import AlternatingLeastSquares
lib = AlternatingLeastSquares(factors=64, regularization=0.05, alpha=ru.ALPHA,
                              iterations=15, random_state=42)
lib.fit(R, show_progress=False)
lib_scores = ru.score_als(lib.user_factors, lib.item_factors, users, cands)
r_lib, n_lib = ru.score_metrics(lib_scores, k=10)

item_pop = np.asarray(R.getnnz(axis=0)).ravel().astype(np.float32)
r_pop, n_pop = ru.score_metrics(ru.score_popularity(item_pop, cands), k=10)
print(f'{"Model":<24}{"Recall@10":>12}{"NDCG@10":>12}')
print('-'*48)
for name, r, n in [('Popularity baseline', r_pop, n_pop),
                   ('from-scratch WMF (mine)', r_scratch, n_scratch),
                   ('implicit ALS (library)', r_lib, n_lib)]:
    print(f'{name:<24}{r:>12.4f}{n:>12.4f}')

/home/racloop/Documents/Personal/coursera/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model                      Recall@10     NDCG@10
------------------------------------------------
Popularity baseline           0.8064      0.5285
from-scratch WMF (mine)       0.9259      0.6970
implicit ALS (library)        0.9275      0.6961


**Takeaway:** the from-scratch ALS lands essentially on top of the library on the same data,
confirming the implementation of the HKV closed-form update is correct — and it beats the
popularity baseline, which is the bar a real personalized model must clear.